In [ ]:
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor
from scipy import stats
import pandas as pd
import xskillscore as xs
import dask
import logging

from pathlib import Path

#import albedo_functions as af

In [ ]:
# Configurazione logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),  # Write to file
    ]
)

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
era_var='2t'

# Percorso ai dati
DATA_PATH = POST_DATA
OUTPUT_PATH = FIG_DIR_2025
OBS_PATH = WORK_DIR

In [ ]:
def bootstrap_quantile_bias(sens, ctrl, ref, iterations=1000):
#    sens = sens.chunk({'member':-1,'time': -1, 'lon': -1, 'lat': -1})
#    ctrl = ctrl.chunk({'member':-1,'time': -1, 'lon': -1, 'lat':-1})
#    ref = ref.chunk({'time': -1, 'lon': -1, 'lat': -1})
    with dask.config.set(**{'array.slicing.split_large_chunks': True}):
        f = xr.concat([sens, ctrl], dim='member').chunk({'member':-1})
        f_ra = xs.resampling.resample_iterations(f, iterations, 'member', replace=True).mean('member').squeeze().compute()
        f_rb = xs.resampling.resample_iterations(f, iterations, 'member', replace=True).mean('member').squeeze().compute()
        bias_f_ra = f_ra.mean('time') - ref
        bias_f_rb = f_rb.mean('time') - ref
        delta_bias = bias_f_ra - bias_f_rb
        sig_delta = delta_bias.chunk(dict(iteration=-1)).quantile([0.025,0.05,0.10,0.90,0.95,0.975], dim='iteration')
    return sig_delta

In [ ]:
def process_lead_years(exp_ctrl, exp_sens, var, y1, y2, save_path):
    """Elabora dati per un dato periodo di lead year."""
    lead = f"{y1}-{y2}"
    lead_number = y2 - y1 + 1
    logging.info(f"Starting main {lead}")
    try:
        logging.info(f"Carica dataset {lead}")
        # Caricamento dataset di controllo
        dset_ctrl_path = DATA_PATH / exp_ctrl / "1x1" / var / f"{exp_ctrl}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_{lead}_1x1_ensemble_rad.nc"
        dset_ctrl = xr.open_dataset(dset_ctrl_path)
        dset_ctrl['time'] = pd.to_datetime(dset_ctrl['time'].values).normalize().to_period('M').start_time
        
        # Caricamento dataset sensibile
        dset_sens_path = DATA_PATH / exp_sens / "1x1" / var / f"{exp_sens}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_{lead}_1x1_ensemble_rad.nc"
        dset_sens = xr.open_dataset(dset_sens_path)
        dset_sens['time'] = pd.to_datetime(dset_sens['time'].values).normalize().to_period('M').start_time
        
        # Caricamento osservazioni
        obs_path = OBS_PATH / f"ERA5_{era_var}_1x1_{lead_number}year.nc"
        obs = xr.open_dataset(obs_path)
        if lead_number in [2,4]:
            obs['time'] = pd.to_datetime(obs['time'].values) - pd.DateOffset(months=1)
        obs['time'] = pd.to_datetime(obs['time'].values).normalize().to_period('M').start_time
        #obs = obs.rename({'bb_albedo': 'alb'})
        obs = obs.rename({f'{era_var}': f'{var}'})
        obs = obs.sel(time=slice(dset_ctrl.time[0], dset_ctrl.time[-1]))
        
        #select time from 1999
        start_date = '1999-09-01'
        
        dset_ctrl_1999 = dset_ctrl.sel(time=slice(start_date, None))
        dset_sens_1999 = dset_sens.sel(time=slice(start_date, None))
        obs_1999 = obs.sel(time=slice(start_date, None))
        
        # Calcolo della differenza per ogni punto e per ogni tempo
        bias_ctrl_time_series = (dset_ctrl[var].mean('member') - obs[var])
        bias_ctrl = bias_ctrl_time_series.mean('time')
        
        bias_sens_time_series = (dset_sens[var].mean('member') - obs[var])
        bias_sens = bias_sens_time_series.mean('time')

        logging.info(f"T-Test {lead}")
        # Applichiamo il test t lungo la dimensione 'time'
        _, ctrl_p_value = stats.ttest_1samp(bias_ctrl_time_series, popmean=0, axis=0)
        _, sens_p_value = stats.ttest_1samp(bias_sens_time_series, popmean=0, axis=0)

        ctrl_p_value = xr.DataArray(
            data=ctrl_p_value,
            coords=bias_ctrl.coords,
            dims=["lat", "lon"],
            name="p"
        )

        sens_p_value = xr.DataArray(
            data=sens_p_value,
            coords=bias_sens.coords,
            dims=["lat", "lon"],
            name="p"
        )
        logging.info(f"Starting bootstrap {lead}")
        delta = bootstrap_quantile_bias(dset_ctrl, dset_sens, obs.mean('time'))
        logging.info(f"fine bootstrap {lead}")
        
        # Save results
        ctrl_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_bias.nc'
        sens_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_bias.nc'
        ctrl_p_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_bias_p.nc'
        sens_p_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_bias_p.nc'
        delta_outfile = f'{save_path}/delta_{var}_lead_{lead}_bias_quantile.nc'
        
        bias_ctrl.to_dataset(name=var).to_netcdf(ctrl_outfile)
        bias_sens.to_dataset(name=var).to_netcdf(sens_outfile)
        ctrl_p_value.to_netcdf(ctrl_p_outfile)
        sens_p_value.to_netcdf(sens_p_outfile)
        delta.to_netcdf(delta_outfile)

        logging.info(f"file salvati {lead}")

    except Exception as e:
        logging.exception(f"Error occurred for lead {lead}: {e}")

In [ ]:
%%time
def main(exp_ctrl, exp_sens, var):
    """Funzione principale per elaborare i dati in parallelo."""
    futures = []
    with concurrent.futures.ProcessPoolExecutor(max_workers=2) as executor:
        for y1 in range(5):
            for y2 in range(y1 + 1, 5):
                logging.info(f"Submitting task for lead years {y1}-{y2}")
                futures.append(
                    executor.submit(process_lead_years, exp_ctrl, exp_sens, var, y1, y2, OBS_PATH)
                )
        # Wait for all tasks to complete
        # progresso: stampa ogni task man mano che termina
        for _i, _f in enumerate(concurrent.futures.as_completed(futures), 1):
            _f.result()
            print(f"  completato {_i}/{len(futures)}", flush=True)

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    
    # Define or import exp_ctrl, exp_sens, var, and process_lead_years
    main(exp_ctrl, exp_sens, var)